# 1. Setting Up for Baseline Modeling

In [7]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# 2. Creating and Saving the Final Preprocessor

In [8]:
# Loading the clean data
df = pd.read_csv('data/processed/cleaned_laptops.csv')

# Defining features (X) and target (y)
X = df.drop(columns=['price', 'name'])
y = df['price']

# Identifying column types from our new clean data
numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Creating the final preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

# --- THE CRITICAL FIX ---
# Fitting the preprocessor on the ENTIRE dataset (X) for consistency
print("Fitting the final preprocessor on the entire dataset...")
preprocessor.fit(X)

# Saving the correctly fitted preprocessor for use in other notebooks and the app
os.makedirs('models', exist_ok=True)
joblib.dump(preprocessor, 'models/preprocessor.joblib')
print("Final preprocessor saved successfully.")

Fitting the final preprocessor on the entire dataset...
Final preprocessor saved successfully.


# 3. Comparing Baseline Models (for Report)

In [9]:
# This part is for generating the comparison table for your report
print("\nComparing baseline models...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results[name] = {'R-squared': r2_score(y_test, y_pred), 'MAE': mean_absolute_error(y_test, y_pred)}

results_df = pd.DataFrame(results).T
print(results_df)


Comparing baseline models...
                   R-squared           MAE
Linear Regression   0.597603  23937.182297
Random Forest       0.751175  17564.092832
